# BERTimbau Large + AWP + Optuna

Combina as melhores tecnicas identificadas ate 2026-04-17:

- Modelo: `neuralmind/bert-large-portuguese-cased`
- MAX_LEN=512 (+2.4% vs 192, confirmado)
- 5-Fold Stratified CV
- Focal Loss (gamma=2.0, alpha=0.25)
- Label Smoothing leve (0.05)
- **AWP** (Adversarial Weight Perturbation, novo)
- **Optuna** calibration 300 trials (temperatura + 7 thresholds simultaneos, novo)
- FP16 Mixed Precision

**Score de referencia:** 0.84027 (BERTimbau 5-fold MAX_LEN=512 sem AWP/Optuna)

**Kaggle Input necessario:** `neuralmind/bert-large-portuguese-cased` (dataset offline)

In [ ]:
# ============================================================
# CONFIGURACAO
# ============================================================
import os

# Caminhos confirmados em 2026-04-17
MODEL_PATH = "/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1"
TRAIN_PATH = "/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv"
TEST_PATH  = "/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv"

MAX_LEN = 512
BATCH_SIZE = 8
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS = 5
SEED = 42

# Focal Loss
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
LABEL_SMOOTHING = 0.05

# AWP
USE_AWP = True
AWP_LR = 1e-1
AWP_EPS = 1e-2
AWP_START_EPOCH = 1   # Ativa AWP a partir da epoch 2 (warmup na 1a)

# Optuna
N_TRIALS = 300

USE_FP16 = True
NUM_CLASSES = 7

print(f"MAX_LEN={MAX_LEN}, EPOCHS={EPOCHS}, LR={LR}")
print(f"AWP={USE_AWP}, Optuna trials={N_TRIALS}")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import optuna
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# DADOS
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(train_df.columns.tolist())
print(train_df.head(2))

# Identificar coluna de texto (ajustar se necessario)
TEXT_COL   = 'text'
TARGET_COL = 'target'

train_texts  = train_df[TEXT_COL].fillna('').tolist()
train_labels = train_df[TARGET_COL].tolist()
test_texts   = test_df[TEXT_COL].fillna('').tolist()

print(f"\nDistribuicao de classes:")
print(train_df[TARGET_COL].value_counts().sort_index())

In [ ]:
# ============================================================
# TOKENIZADOR
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True,   # OBRIGATORIO no Kaggle offline
)
print(f"Vocab size: {tokenizer.vocab_size}")

In [ ]:
# ============================================================
# DATASET
# ============================================================
class SPRDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=MAX_LEN):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding='max_length',
            return_tensors='pt',
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc['token_type_ids'].squeeze(),  # BERTimbau usa token_type_ids
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [ ]:
# ============================================================
# FOCAL LOSS COM LABEL SMOOTHING
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA,
                 label_smoothing=LABEL_SMOOTHING, num_classes=NUM_CLASSES):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing
        self.num_classes     = num_classes

    def forward(self, inputs, targets):
        # Label smoothing manual para poder combinar com focal
        if self.label_smoothing > 0:
            smooth = self.label_smoothing / self.num_classes
            one_hot = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
            one_hot = one_hot * (1 - self.label_smoothing) + smooth
            log_prob = F.log_softmax(inputs, dim=-1)
            ce_loss  = -(one_hot * log_prob).sum(dim=-1)
        else:
            ce_loss = F.cross_entropy(inputs, targets, reduction='none')

        pt         = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

In [ ]:
# ============================================================
# AWP (Adversarial Weight Perturbation)
# Perturba pesos na direcao do gradiente -> mínimos mais planos -> melhor generalizacao
# Referencia: https://arxiv.org/abs/2004.05884
# ============================================================
class AWP:
    def __init__(self, model, optimizer, scaler, awp_lr=AWP_LR, awp_eps=AWP_EPS):
        self.model     = model
        self.optimizer = optimizer
        self.scaler    = scaler
        self.awp_lr    = awp_lr
        self.awp_eps   = awp_eps
        self.backup    = {}
        self.backup_eps = {}

    def _save(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0 and not torch.isnan(norm):
                    self.backup_eps[name] = self.awp_eps / norm

    def _attack_step(self):
        e = 1e-6
        for name, param in self.model.named_parameters():
            if param.requires_grad and param.grad is not None and name in self.backup_eps:
                eps = self.backup_eps[name]
                param.data = param.data + self.awp_lr * param.grad / (torch.norm(param.grad) + e)
                param.data = torch.min(
                    torch.max(param.data, self.backup[name] - eps),
                    self.backup[name] + eps
                )

    def _restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]
        self.backup = {}
        self.backup_eps = {}

    def attack_backward(self, batch, criterion):
        """Executa passo adversarial. Chamar APOS loss.backward() normal."""
        self._save()
        self._attack_step()
        self.model.zero_grad()
        labels  = batch['labels'].to(device)
        inputs  = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        with autocast(dtype=torch.float16, enabled=USE_FP16):
            outputs  = self.model(**inputs)
            adv_loss = criterion(outputs.logits, labels)
        self.scaler.scale(adv_loss).backward()
        self._restore()

In [ ]:
# ============================================================
# TRAINING E INFERENCE
# ============================================================
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion, awp=None, epoch=0):
    model.train()
    total_loss = 0.0
    for batch in loader:
        labels = batch['labels'].to(device)
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}

        optimizer.zero_grad()
        with autocast(dtype=torch.float16, enabled=USE_FP16):
            outputs = model(**inputs)
            loss    = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()

        # AWP adversarial step
        if awp is not None and epoch >= AWP_START_EPOCH:
            awp.attack_backward(batch, criterion)

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        inputs = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
        with autocast(dtype=torch.float16, enabled=USE_FP16):
            outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)


def compute_f1(probs, labels):
    preds = np.argmax(probs, axis=1)
    return f1_score(labels, preds, average='macro')

In [ ]:
# ============================================================
# 5-FOLD CROSS VALIDATION
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train_labels_arr = np.array(train_labels)

oof_probs   = np.zeros((len(train_texts), NUM_CLASSES))
test_probs  = np.zeros((len(test_texts),  NUM_CLASSES))
fold_scores = []

criterion = FocalLoss()

for fold, (train_idx, val_idx) in enumerate(skf.split(train_texts, train_labels_arr)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold + 1}/{N_FOLDS}")
    print(f"{'='*50}")

    # Dados do fold
    tr_texts  = [train_texts[i]  for i in train_idx]
    tr_labels = [train_labels[i] for i in train_idx]
    va_texts  = [train_texts[i]  for i in val_idx]
    va_labels = [train_labels[i] for i in val_idx]

    tr_ds   = SPRDataset(tr_texts, tr_labels, tokenizer)
    va_ds   = SPRDataset(va_texts, va_labels, tokenizer)
    te_ds   = SPRDataset(test_texts, None, tokenizer)

    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    va_dl = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    te_dl = DataLoader(te_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Modelo
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        num_labels=NUM_CLASSES,
        local_files_only=True,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    total_steps  = len(tr_dl) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = GradScaler(enabled=USE_FP16)
    awp    = AWP(model, optimizer, scaler) if USE_AWP else None

    best_val_f1    = 0.0
    best_val_probs = None
    best_te_probs  = None

    for epoch in range(EPOCHS):
        tr_loss = train_epoch(model, tr_dl, optimizer, scheduler, scaler, criterion, awp, epoch)

        va_probs = predict(model, va_dl)
        val_f1   = compute_f1(va_probs, va_labels)

        awp_str = f" [AWP ativo]" if (USE_AWP and epoch >= AWP_START_EPOCH) else ""
        print(f"Epoch {epoch+1}/{EPOCHS} | loss={tr_loss:.4f} | val_f1={val_f1:.5f}{awp_str}")

        if val_f1 > best_val_f1:
            best_val_f1    = val_f1
            best_val_probs = va_probs.copy()
            best_te_probs  = predict(model, te_dl)
            print(f"  -> Novo melhor! F1={best_val_f1:.5f}")

    oof_probs[val_idx] = best_val_probs
    test_probs += best_te_probs / N_FOLDS
    fold_scores.append(best_val_f1)

    del model, optimizer, scheduler, scaler, awp
    torch.cuda.empty_cache()

oof_f1_raw = compute_f1(oof_probs, train_labels_arr)
print(f"\nCV F1-macro: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
print(f"OOF F1-macro (argmax): {oof_f1_raw:.5f}")

In [ ]:
# ============================================================
# OPTUNA: CALIBRACAO CONJUNTA (temperatura + thresholds)
# Otimiza simultaneamente 1 temperatura escalar + 7 multiplicadores de threshold
# Muito mais robusto que grid search sequencial
# ============================================================
def apply_calibration(probs, temp, thresholds):
    """Escala logits por temperatura e aplica multiplicadores de threshold."""
    calibrated = probs ** (1.0 / temp)
    calibrated = calibrated / calibrated.sum(axis=1, keepdims=True)
    # Multiplicadores: amplia ou reduz a confianca por classe
    scaled = calibrated * np.array(thresholds)
    return np.argmax(scaled, axis=1)


def objective(trial):
    temp = trial.suggest_float('temp', 0.5, 2.0)
    thresholds = [
        trial.suggest_float(f'thr_{c}', 0.3, 3.0)
        for c in range(NUM_CLASSES)
    ]
    preds = apply_calibration(oof_probs, temp, thresholds)
    return f1_score(train_labels_arr, preds, average='macro')


optuna.logging.set_verbosity(optuna.logging.WARNING)
sampler = optuna.samplers.TPESampler(seed=SEED)
study   = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_temp       = study.best_params['temp']
best_thresholds = [study.best_params[f'thr_{c}'] for c in range(NUM_CLASSES)]
best_oof_f1     = study.best_value

print(f"\nOptuna ({N_TRIALS} trials):")
print(f"  Temperatura: {best_temp:.4f}")
print(f"  Thresholds:  {[f'{t:.4f}' for t in best_thresholds]}")
print(f"  OOF F1 baseline (argmax): {oof_f1_raw:.5f}")
print(f"  OOF F1 calibrado (Optuna): {best_oof_f1:.5f}")
print(f"  Ganho: +{(best_oof_f1 - oof_f1_raw)*100:.3f} pp")

In [ ]:
# ============================================================
# SUBMISSAO
# ============================================================
test_preds = apply_calibration(test_probs, best_temp, best_thresholds)

submission = pd.DataFrame({
    'id':     test_df['id'],
    'target': test_preds,
})
submission.to_csv('submission.csv', index=False)

print("submission.csv gerado!")
print(submission['target'].value_counts().sort_index())
print(f"\nResumo:")
print(f"  Modelo: BERTimbau Large")
print(f"  MAX_LEN: {MAX_LEN}")
print(f"  AWP: {USE_AWP}")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  OOF F1 (argmax):  {oof_f1_raw:.5f}")
print(f"  OOF F1 (Optuna):  {best_oof_f1:.5f}")